<a href="https://colab.research.google.com/github/AshokGit544/Hugging-Face-Project/blob/main/Hugging_Face_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers datasets sentence-transformers scikit-learn pandas numpy

In [2]:
import os
import re
import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from datasets import Dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
BASE_PATH = Path("/content/enterprise_hf_project")
DATA_PATH = BASE_PATH / "data"
OUTPUT_PATH = BASE_PATH / "output"
CHUNK_PATH = BASE_PATH / "chunks"

for path in [BASE_PATH, DATA_PATH, OUTPUT_PATH, CHUNK_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully")

Project folders created successfully


In [4]:
random.seed(42)

document_types = [
    "Supplier Issue Report",
    "Maintenance Log",
    "Quality Incident",
    "Finance Memo",
    "Engineering Change Notice"
]

suppliers = ["Bosch", "Siemens", "Denso", "Magna", "Valeo"]
plants = ["PLT100", "PLT200", "PLT300"]
equipment_ids = ["EQP1001", "EQP1002", "EQP1003", "EQP2001", "EQP2002"]
company_codes = ["C001", "C002", "C003"]
issues = [
    "sensor failure causing unexpected downtime",
    "voltage fluctuation impacting assembly line stability",
    "invoice mismatch for replacement material",
    "quality defect identified during inspection",
    "maintenance delay due to unavailable spare parts",
    "temperature alarm triggered in production unit",
    "supplier delivery issue impacting scheduled repairs",
    "equipment shutdown linked to recurring controller fault"
]

documents = []

for i in range(1, 51):
    supplier = random.choice(suppliers)
    plant = random.choice(plants)
    equipment = random.choice(equipment_ids)
    company = random.choice(company_codes)
    doc_type = random.choice(document_types)
    issue = random.choice(issues)
    amount = random.randint(1000, 25000)
    date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    common_key = f"{plant}_{equipment}"

    text = (
        f"{doc_type} for supplier {supplier} at plant {plant}. "
        f"Equipment {equipment} reported {issue}. "
        f"Document date is {date}. "
        f"Invoice amount recorded was {amount} USD. "
        f"Company code {company}. "
        f"Common business key {common_key}. "
        f"This document supports manufacturing operations, supplier review, "
        f"issue tracking, finance review, and engineering analysis."
    )

    documents.append({
        "document_id": f"DOC{i:03d}",
        "document_type": doc_type,
        "company_code": company,
        "plant_code": plant,
        "supplier_name": supplier,
        "equipment_id": equipment,
        "document_date": date,
        "invoice_amount": amount,
        "common_business_key": common_key,
        "document_text": text
    })

documents_df = pd.DataFrame(documents)
documents_df.to_csv(DATA_PATH / "enterprise_documents.csv", index=False)
documents_df.head()

,document_id,document_type,company_code,plant_code,supplier_name,equipment_id,document_date,invoice_amount,common_business_key,document_text
0,DOC001,Maintenance Log,C001,PLT100,Bosch,EQP1003,2024-11-24,4358,PLT100_EQP1003,Maintenance Log for supplier Bosch at plant PL...
1,DOC002,Supplier Issue Report,C002,PLT100,Valeo,EQP2002,2024-04-08,4070,PLT100_EQP2002,Supplier Issue Report for supplier Valeo at pl...
2,DOC003,Maintenance Log,C003,PLT300,Valeo,EQP1001,2024-08-19,8223,PLT300_EQP1001,Maintenance Log for supplier Valeo at plant PL...
3,DOC004,Finance Memo,C003,PLT100,Denso,EQP1002,2024-03-07,10105,PLT100_EQP1002,Finance Memo for supplier Denso at plant PLT10...
4,DOC005,Supplier Issue Report,C002,PLT100,Denso,EQP1001,2024-10-09,12270,PLT100_EQP1001,Supplier Issue Report for supplier Denso at pl...


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-_\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

documents_df["normalized_text"] = documents_df["document_text"].apply(clean_text)
documents_df.head()

,document_id,document_type,company_code,plant_code,supplier_name,equipment_id,document_date,invoice_amount,common_business_key,document_text,normalized_text
0,DOC001,Maintenance Log,C001,PLT100,Bosch,EQP1003,2024-11-24,4358,PLT100_EQP1003,Maintenance Log for supplier Bosch at plant PL...,maintenance log for supplier bosch at plant pl...
1,DOC002,Supplier Issue Report,C002,PLT100,Valeo,EQP2002,2024-04-08,4070,PLT100_EQP2002,Supplier Issue Report for supplier Valeo at pl...,supplier issue report for supplier valeo at pl...
2,DOC003,Maintenance Log,C003,PLT300,Valeo,EQP1001,2024-08-19,8223,PLT300_EQP1001,Maintenance Log for supplier Valeo at plant PL...,maintenance log for supplier valeo at plant pl...
3,DOC004,Finance Memo,C003,PLT100,Denso,EQP1002,2024-03-07,10105,PLT100_EQP1002,Finance Memo for supplier Denso at plant PLT10...,finance memo for supplier denso at plant plt10...
4,DOC005,Supplier Issue Report,C002,PLT100,Denso,EQP1001,2024-10-09,12270,PLT100_EQP1001,Supplier Issue Report for supplier Denso at pl...,supplier issue report for supplier denso at pl...


In [6]:
hf_dataset = Dataset.from_pandas(
    documents_df[
        [
            "document_id",
            "document_type",
            "company_code",
            "plant_code",
            "supplier_name",
            "equipment_id",
            "document_date",
            "invoice_amount",
            "common_business_key",
            "normalized_text"
        ]
    ]
)

hf_dataset

Dataset({
    features: ['document_id', 'document_type', 'company_code', 'plant_code', 'supplier_name', 'equipment_id', 'document_date', 'invoice_amount', 'common_business_key', 'normalized_text'],
    num_rows: 50
})

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample_tokens = tokenizer(
    hf_dataset[0]["normalized_text"],
    truncation=True,
    padding="max_length",
    max_length=64
)

print("Input IDs length:", len(sample_tokens["input_ids"]))
print("First 20 token IDs:", sample_tokens["input_ids"][:20])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input IDs length: 64
First 20 token IDs: [101, 6032, 8833, 2005, 17024, 25936, 2012, 3269, 20228, 2102, 18613, 1012, 3941, 1041, 4160, 2361, 18613, 2509, 2988, 1999]


In [8]:
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

classification_output = classifier(documents_df["document_text"].head(5).tolist())
classification_output

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'NEGATIVE', 'score': 0.9916297793388367},
 {'label': 'NEGATIVE', 'score': 0.9958750605583191},
 {'label': 'NEGATIVE', 'score': 0.9604532122612},
 {'label': 'NEGATIVE', 'score': 0.9646630883216858},
 {'label': 'NEGATIVE', 'score': 0.9518764019012451}]

In [9]:
ner_pipe = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

ner_results = ner_pipe(
    "Supplier Bosch reported sensor failure at plant PLT200 on equipment EQP1002 dated 2024-05-16."
)

ner_results

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[{'entity_group': 'ORG',
  'score': np.float32(0.9950055),
  'word': 'Supplier Bosch',
  'start': 0,
  'end': 14}]

In [10]:
def chunk_text(text, chunk_size=35):
    words = text.split()
    return [" ".join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]

chunk_rows = []

for _, row in documents_df.iterrows():
    chunks = chunk_text(row["normalized_text"], chunk_size=35)
    for idx, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            "document_id": row["document_id"],
            "chunk_id": f"{row['document_id']}_CHUNK_{idx}",
            "document_type": row["document_type"],
            "company_code": row["company_code"],
            "plant_code": row["plant_code"],
            "supplier_name": row["supplier_name"],
            "equipment_id": row["equipment_id"],
            "document_date": row["document_date"],
            "common_business_key": row["common_business_key"],
            "chunk_text": chunk
        })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df.to_csv(CHUNK_PATH / "chunked_documents.csv", index=False)
chunk_df.head()

,document_id,chunk_id,document_type,company_code,plant_code,supplier_name,equipment_id,document_date,common_business_key,chunk_text
0,DOC001,DOC001_CHUNK_1,Maintenance Log,C001,PLT100,Bosch,EQP1003,2024-11-24,PLT100_EQP1003,maintenance log for supplier bosch at plant pl...
1,DOC001,DOC001_CHUNK_2,Maintenance Log,C001,PLT100,Bosch,EQP1003,2024-11-24,PLT100_EQP1003,supports manufacturing operations supplier rev...
2,DOC002,DOC002_CHUNK_1,Supplier Issue Report,C002,PLT100,Valeo,EQP2002,2024-04-08,PLT100_EQP2002,supplier issue report for supplier valeo at pl...
3,DOC002,DOC002_CHUNK_2,Supplier Issue Report,C002,PLT100,Valeo,EQP2002,2024-04-08,PLT100_EQP2002,document supports manufacturing operations sup...
4,DOC003,DOC003_CHUNK_1,Maintenance Log,C003,PLT300,Valeo,EQP1001,2024-08-19,PLT300_EQP1001,maintenance log for supplier valeo at plant pl...


In [11]:
chunk_df["embedding_ready_text"] = (
    "DocumentType: " + chunk_df["document_type"] + " | " +
    "CompanyCode: " + chunk_df["company_code"] + " | " +
    "Plant: " + chunk_df["plant_code"] + " | " +
    "Supplier: " + chunk_df["supplier_name"] + " | " +
    "Equipment: " + chunk_df["equipment_id"] + " | " +
    "Date: " + chunk_df["document_date"] + " | " +
    "CommonKey: " + chunk_df["common_business_key"] + " | " +
    "Text: " + chunk_df["chunk_text"]
)

chunk_df.head()

,document_id,chunk_id,document_type,company_code,plant_code,supplier_name,equipment_id,document_date,common_business_key,chunk_text,embedding_ready_text
0,DOC001,DOC001_CHUNK_1,Maintenance Log,C001,PLT100,Bosch,EQP1003,2024-11-24,PLT100_EQP1003,maintenance log for supplier bosch at plant pl...,DocumentType: Maintenance Log | CompanyCode: C...
1,DOC001,DOC001_CHUNK_2,Maintenance Log,C001,PLT100,Bosch,EQP1003,2024-11-24,PLT100_EQP1003,supports manufacturing operations supplier rev...,DocumentType: Maintenance Log | CompanyCode: C...
2,DOC002,DOC002_CHUNK_1,Supplier Issue Report,C002,PLT100,Valeo,EQP2002,2024-04-08,PLT100_EQP2002,supplier issue report for supplier valeo at pl...,DocumentType: Supplier Issue Report | CompanyC...
3,DOC002,DOC002_CHUNK_2,Supplier Issue Report,C002,PLT100,Valeo,EQP2002,2024-04-08,PLT100_EQP2002,document supports manufacturing operations sup...,DocumentType: Supplier Issue Report | CompanyC...
4,DOC003,DOC003_CHUNK_1,Maintenance Log,C003,PLT300,Valeo,EQP1001,2024-08-19,PLT300_EQP1001,maintenance log for supplier valeo at plant pl...,DocumentType: Maintenance Log | CompanyCode: C...


In [12]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunk_df["embedding_ready_text"].tolist(),
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding shape: (100, 384)


In [13]:
query = "Find supplier related sensor failure documents for plant PLT200 and equipment EQP1002 with finance or issue context"

query_embedding = embedding_model.encode([query])

scores = cosine_similarity(query_embedding, embeddings).flatten()
top_indices = scores.argsort()[-10:][::-1]

search_results = chunk_df.iloc[top_indices].copy()
search_results["similarity_score"] = scores[top_indices]

search_results = search_results[
    [
        "chunk_id",
        "document_id",
        "document_type",
        "company_code",
        "plant_code",
        "supplier_name",
        "equipment_id",
        "document_date",
        "common_business_key",
        "chunk_text",
        "similarity_score"
    ]
]

search_results.to_csv(OUTPUT_PATH / "semantic_search_results.csv", index=False)
search_results.head(10)

,chunk_id,document_id,document_type,company_code,plant_code,supplier_name,equipment_id,document_date,common_business_key,chunk_text,similarity_score
2,DOC002_CHUNK_1,DOC002,Supplier Issue Report,C002,PLT100,Valeo,EQP2002,2024-04-08,PLT100_EQP2002,supplier issue report for supplier valeo at pl...,0.747814
12,DOC007_CHUNK_1,DOC007,Supplier Issue Report,C001,PLT200,Valeo,EQP2002,2024-04-25,PLT200_EQP2002,supplier issue report for supplier valeo at pl...,0.738327
66,DOC034_CHUNK_1,DOC034,Quality Incident,C001,PLT300,Bosch,EQP2002,2024-10-16,PLT300_EQP2002,quality incident for supplier bosch at plant p...,0.735197
20,DOC011_CHUNK_1,DOC011,Maintenance Log,C001,PLT100,Valeo,EQP1003,2024-07-09,PLT100_EQP1003,maintenance log for supplier valeo at plant pl...,0.721993
38,DOC020_CHUNK_1,DOC020,Maintenance Log,C002,PLT100,Bosch,EQP1003,2024-10-03,PLT100_EQP1003,maintenance log for supplier bosch at plant pl...,0.719184
22,DOC012_CHUNK_1,DOC012,Quality Incident,C003,PLT100,Bosch,EQP2002,2024-08-13,PLT100_EQP2002,quality incident for supplier bosch at plant p...,0.717614
48,DOC025_CHUNK_1,DOC025,Supplier Issue Report,C001,PLT300,Bosch,EQP1001,2024-02-17,PLT300_EQP1001,supplier issue report for supplier bosch at pl...,0.714262
46,DOC024_CHUNK_1,DOC024,Supplier Issue Report,C002,PLT100,Siemens,EQP1001,2024-04-01,PLT100_EQP1001,supplier issue report for supplier siemens at ...,0.704697
36,DOC019_CHUNK_1,DOC019,Engineering Change Notice,C001,PLT100,Siemens,EQP1003,2024-06-16,PLT100_EQP1003,engineering change notice for supplier siemens...,0.697966
94,DOC048_CHUNK_1,DOC048,Maintenance Log,C003,PLT200,Valeo,EQP2002,2024-03-14,PLT200_EQP2002,maintenance log for supplier valeo at plant pl...,0.696106


In [14]:
model_name = "google-t5/t5-small"

sum_tokenizer = AutoTokenizer.from_pretrained(model_name)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

sample_context = " ".join(search_results["chunk_text"].head(3).tolist())
sample_context = sample_context[:1200]

prompt = f"summarize: {sample_context}"

inputs = sum_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

output_ids = sum_model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False
)

summary_text = sum_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(summary_text)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

equipment eqp2002 reported sensor failure causing unexpected downtime. document date is 2024-04-08. invoice amount recorded was 4070 usd.


In [15]:
summary_json = {
    "generated_at": datetime.now().isoformat(),
    "document_count": int(len(documents_df)),
    "chunk_count": int(len(chunk_df)),
    "embedding_count": int(len(embeddings)),
    "sample_query": query,
    "top_result_document_id": search_results.iloc[0]["document_id"] if len(search_results) > 0 else None,
    "sample_summary": summary_text
}

with open(OUTPUT_PATH / "project_summary.json", "w") as f:
    json.dump(summary_json, f, indent=2)

print(json.dumps(summary_json, indent=2))

{
  "generated_at": "2026-03-29T14:23:35.037292",
  "document_count": 50,
  "chunk_count": 100,
  "embedding_count": 100,
  "sample_query": "Find supplier related sensor failure documents for plant PLT200 and equipment EQP1002 with finance or issue context",
  "top_result_document_id": "DOC002",
  "sample_summary": "equipment eqp2002 reported sensor failure causing unexpected downtime. document date is 2024-04-08. invoice amount recorded was 4070 usd."
}


In [16]:
print("Saved files:")
for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        print(os.path.join(root, file))

Saved files:
/content/enterprise_hf_project/data/enterprise_documents.csv
/content/enterprise_hf_project/output/semantic_search_results.csv
/content/enterprise_hf_project/output/project_summary.json
/content/enterprise_hf_project/chunks/chunked_documents.csv
